# **Optimal Travel Window Predictor**

**Project Objective**
The goal of this project is to predict the optimal time of year to visit a specific destination, here,Lisbon, Portugal. Real-world data rarely aligns perfectly, so this notebook merges two asynchronous datasets—historical hotel bookings (2015-2017) and global climate data (2024-2026), by engineering an "Annual Profile."

The model uses a Random Forest Classifier to identify prime travel windows based on crowd density thresholds, temperature ranges, and precipitation levels.

**Hotel Bookings Dataset**

Out of the original 32 columns, we aggressively filtered down to just these 7 essential features to measure actual historical crowd sizes and pricing.

***arrival_date_month***: The month of the year the guest was scheduled to arrive.

***arrival_date_day_of_month***: The specific day of the month the guest arrived.

***is_canceled***: A binary column (0 or 1) indicating if the booking was canceled.

***adults***: The number of adults on the reservation.

***children***: The number of children on the reservation.

***babies***: The number of babies on the reservation.

***adr***: Stands for Average Daily Rate. This is the average price the guest paid per room, per night.

**Global Weather Repository**

Out of the original 41 columns, we filtered down to these 4 critical features to measure environmental comfort.

***location_name***: The specific city where the weather data was recorded.

***last_updated***: A timestamp object showing exactly when the weather was recorded.

***temperature_celsius:*** The recorded temperature in degrees Celsius.

***precip_mm***: The amount of precipitation (rainfall) recorded in millimeters.

In [40]:
# Importing  required Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [41]:
# Data Ingestion with only those columns that our model will need
df_hotel = pd.read_csv('hotel_bookings.csv', usecols=['arrival_date_month', 'arrival_date_day_of_month', 'is_canceled', 'adults', 'children', 'babies', 'adr'])
df_weather = pd.read_csv('GlobalWeatherRepository.csv', usecols=['location_name', 'last_updated', 'temperature_celsius', 'precip_mm'])

In [42]:
# Displaying Hotel Dataset
df_hotel

,is_canceled,arrival_date_month,arrival_date_day_of_month,adults,children,babies,adr
0,0,July,1,2,0.0,0,0.00
1,0,July,1,2,0.0,0,0.00
2,0,July,1,1,0.0,0,75.00
3,0,July,1,1,0.0,0,75.00
4,0,July,1,2,0.0,0,98.00
...,...,...,...,...,...,...,...
119385,0,August,30,2,0.0,0,96.14
119386,0,August,31,3,0.0,0,225.43
119387,0,August,31,2,0.0,0,157.71
119388,0,August,31,2,0.0,0,104.40


In [43]:
# Displaying Weather Report Dataset
df_weather # this data covers weather report for many places

,location_name,last_updated,temperature_celsius,precip_mm
0,Kabul,2024-05-16 13:15,26.6,0.00
1,Tirana,2024-05-16 10:45,19.0,0.10
2,Algiers,2024-05-16 09:45,23.0,0.00
3,Andorra La Vella,2024-05-16 10:45,6.3,0.30
4,Luanda,2024-05-16 09:45,26.0,0.00
...,...,...,...,...
141698,Caracas,2026-05-16 02:15,18.2,0.00
141699,Hanoi,2026-05-16 13:15,26.1,0.12
141700,Sanaa,2026-05-16 09:15,23.0,0.00
141701,Lusaka,2026-05-16 08:30,18.0,0.00


Cleaning and Processing **HOTEL** Data

In [44]:
# Verify data integrity by checking for any missing values in the filtered dataset
df_hotel.isnull().sum()

,0
is_canceled,0
arrival_date_month,0
arrival_date_day_of_month,0
adults,0
children,4
babies,0
adr,0


In [45]:
# Only 'children' column has missing data
# As the number of missing values is very small we fill them with '0'
df_hotel['children'] = df_hotel['children'].fillna(0)

In [46]:
# Convert string months to numerical values using Pandas vectorized operations
df_hotel['Month'] = pd.to_datetime(df_hotel['arrival_date_month'], format='%B').dt.month
df_hotel['Day'] = df_hotel['arrival_date_day_of_month']

# Dropping columns that we do not need further
df_hotel = df_hotel.drop(['arrival_date_month', 'arrival_date_day_of_month'], axis=1)

In [47]:
# Filter to keep actual visitors (drop cancellations)
df_hotel_actual = df_hotel[df_hotel['is_canceled'] == 0].copy()

# Dropping columns that we do not need further
df_hotel_actual = df_hotel_actual.drop(['is_canceled'], axis=1)

In [48]:
# Calculate total footfall
df_hotel_actual['Total_Guests'] = df_hotel_actual['adults'] + df_hotel_actual['children'] + df_hotel_actual['babies']

# Dropping columns that we do not need further
df_hotel_actual = df_hotel_actual.drop(['adults', 'children', 'babies'], axis=1)

In [49]:
df_hotel_actual.sample(5)

,adr,Month,Day,Total_Guests
31871,130.50,6,2,2.0
36246,123.67,5,12,2.0
93500,115.00,7,21,2.0
87508,126.23,4,17,2.0
19585,104.00,12,26,3.0


Cleaning and Processing **WEATHER** Data

In [50]:
# Since our Hotel data originates from Portugal, we filter the global weather
# dataset to strictly include Lisbon to ensure accurate geographical matching.
df_weather_pt = df_weather[df_weather['location_name'] == 'Lisbon'].copy()
df_weather_pt

,location_name,last_updated,temperature_celsius,precip_mm
140,Lisbon,2024-05-16 09:45,16.0,0.01
335,Lisbon,2024-05-16 15:15,19.0,0.00
529,Lisbon,2024-05-17 17:00,19.0,0.00
721,Lisbon,2024-05-18 15:30,19.0,0.00
916,Lisbon,2024-05-19 15:15,20.0,0.00
...,...,...,...,...
140868,Lisbon,2026-05-12 07:30,14.2,0.09
141063,Lisbon,2026-05-13 07:30,12.1,0.00
141258,Lisbon,2026-05-14 07:30,13.2,0.00
141453,Lisbon,2026-05-15 07:45,13.1,0.00


In [51]:
# Verify data integrity by checking for any missing values in the filtered dataset
df_weather_pt.isnull().sum() # Result confirms no missing data

,0
location_name,0
last_updated,0
temperature_celsius,0
precip_mm,0


In [52]:
# Extract Month and Day from the Datetime object to prepare for dataset alignment
df_weather_pt['Date'] = pd.to_datetime(df_weather_pt['last_updated'])
df_weather_pt['Month'] = df_weather_pt['Date'].dt.month
df_weather_pt['Day'] = df_weather_pt['Date'].dt.day

# Dropping columns that we do not need further
df_weather_pt = df_weather_pt.drop(['last_updated', 'Date'], axis=1)

In [53]:
df_weather_pt.sample(5)

,location_name,temperature_celsius,precip_mm,Month,Day
100109,Lisbon,19.2,0.00,10,13
105178,Lisbon,14.3,0.01,11,8
111613,Lisbon,12.0,0.00,12,11
38174,Lisbon,17.2,0.25,11,29
24777,Lisbon,21.4,0.00,9,21


Create **Annual Crowd Profile** (Hotel Data)

In [54]:
# Group historical data by Month and Day to calculate the average daily footfall
# and pricing across the entire dataset, ignoring the specific year.
daily_crowd = df_hotel_actual.groupby(['Month', 'Day']).agg(
    Daily_Guests=('Total_Guests', 'sum'),
    Average_Price=('adr', 'mean')
).reset_index()

Create **Annual Climate Profile** (Weather Data)

In [55]:
# Group by Month and Day to create our historical weather calendar averages
daily_weather = df_weather_pt.groupby(['Month', 'Day']).agg(
    Temperature=('temperature_celsius', 'mean'),
    Precipitation_mm=('precip_mm', 'mean')
).reset_index()

Data Allignment & Merging

In [56]:
# This perfectly aligns the 2015-2017 hotel data with the 2024-2026 weather data
# by merging on our newly created 'Month' and 'Day' annual profile features.
df_merged = pd.merge(daily_crowd, daily_weather, on=['Month', 'Day'], how='inner')
print(f"--> Successfully created an annual profile with {df_merged.shape[0]} days of matched data.\n")

--> Successfully created an annual profile with 365 days of matched data.



Defining Target Variable Thresholds

In [57]:
# Calculate the "too crowded" threshold (75th percentile of historical daily guests)
crowd_threshold = df_merged['Daily_Guests'].quantile(0.75)

In [58]:
def check_suitability(row):
  """
    Evaluates if a specific day is ideal for travel based on:
    1. Temperature is comfortably between 15°C and 25°C
    2. Minimal rainfall (Under 2.0 mm)
    3. Crowd density is below the 75th percentile
    """
  # Prime Time = 15C to 25C, less than 2mm rain, and NOT in the top 25% of crowds
  good_weather = (15 <= row['Temperature'] <= 25) and (row['Precipitation_mm'] < 2.0)
  not_overcrowded = row['Daily_Guests'] < crowd_threshold

  if good_weather and not_overcrowded:
      return 1 # Suitable
  else:
      return 0 # Not Suitable

In [59]:
# Apply the logic to the dataframe
df_merged['Is_Suitable'] = df_merged.apply(check_suitability, axis=1)

Machine Learning Pipeline Setup

In [60]:
# Isolate the predictive features (X) and the target variable (y)
X = df_merged[['Month', 'Day', 'Temperature', 'Precipitation_mm', 'Average_Price']]
y = df_merged['Is_Suitable']

In [61]:
# Split the data: 80% for training the model, 20% for testing its accuracy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [62]:
# Initialize and train the Random Forest
# Random Forests handle complex, non-linear relationships well
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [63]:
# Generate predictions using the test data
y_predict = model.predict(X_test)

In [64]:
# Calculate and output the model's overall accuracy score in percentage
print('Accuracy Score:',accuracy_score(y_test, y_predict)*100)

Accuracy Score: 87.67123287671232


Save the Trained Model for Delpoyment

In [65]:
import joblib

# Save the model to a file named 'prime_visit_model.pkl'
joblib.dump(model, 'prime_visit_model.pkl')

['prime_visit_model.pkl']